### Step 1: Install Dependencies

In [1]:
! pip install torch transformers datasets accelerate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e0b1f714b860876cafdfe65df946e89f198a01fc4e202316919aa4b8f8cc4d63
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


> You **MUST** work with GPU, so check first:

In [1]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
else:
    print("Please switch to a GPU-enabled environment.")

GPU is available!


### Step 2: Import Libraries

In [2]:
from collections import Counter

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score
from torch.quantization import quantize_dynamic

### Step 3: Dataset Preparation

#### 3.1: Load Data

In [3]:
# Load "lhoestq/conll2003" dataset using HuggingFace datasets library
dataset = load_dataset("lhoestq/conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

#### 3.2: Show Data

In [4]:
# Show samples of data before start working
sample = dataset["train"][0]
print(sample)

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [11]:
from collections import Counter

# check Data Balancing
all_labels = [tag for tags in dataset["train"]["ner_tags"] for tag in tags]
counter = Counter(all_labels)
print(counter)

Counter({0: 169578, 5: 7140, 1: 6600, 3: 6321, 2: 4528, 4: 3704, 7: 3438, 6: 1157, 8: 1155})


#### 3.3: Data Formatting
> Convert dataset into BERT-compatible format
<br>FROM:<br>
{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}
 <br><br>
 TO:<br>
 {'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [101,
  7270,
  22961,
  1528,
  1840,
  1106,
  21423,
  1418,
  2495,
  12913,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]}

Where:
- input_ids will be the X and labels will be the Y
- 101 token id is the <sos> token
- 102 token id is the <eos> token
- -100 is the label for <sos> and <eos>

In [13]:
# use the following tokenizer "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        new_labels = []
        for word_idx in word_ids:
            if word_idx is None:
                new_labels.append(-100)
            else:
                new_labels.append(label[word_idx])
        labels.append(new_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

### Step 4: Model Loading

In [21]:
ner_tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
num_labels = len(ner_tag_names)

In [22]:
# Load pre-trained BERT model with a token classification head
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased",
                                                          num_labels=num_labels)


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [24]:
# Check the model architecture
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

### Step 5: Fine-Tuning

#### 5.1: Hyperparameter Selection

In [25]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir="./logs",
    logging_steps=10,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


#### 5.2: Evaluation Metrices

In [29]:
id2label = {i:label for i, label in enumerate(ner_tag_names)}
label2id = {label:i for i, label in enumerate(ner_tag_names)}

In [36]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_preds = [[id2label[p] for p, l in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]

    precision = precision_score(true_labels, true_preds)
    recall = recall_score(true_labels, true_preds)
    f1 = f1_score(true_labels, true_preds)

    return {"precision": precision, "recall": recall, "f1": f1}

#### 5.3: Training Loop
> ONLY Choose one of the following methods NOT BOTH

In [37]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [38]:
# Default Standard Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator = data_collator,
    compute_metrics=compute_metrics,  # Custom function to calculate F1, precision, recall
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.033322,0.180611,0.879853,0.889322,0.884562
2,0.017004,0.182607,0.897495,0.897579,0.897537
3,0.016797,0.183175,0.895210,0.905093,0.900125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.035667314276879906, metrics={'train_runtime': 521.1843, 'train_samples_per_second': 80.822, 'train_steps_per_second': 5.054, 'total_flos': 1050534559887048.0, 'train_loss': 0.035667314276879906, 'epoch': 3.0})

### Step 6: Save Weights

In [43]:
# Save the model and tokenizer
model.save_pretrained("./ner_model")
tokenizer.save_pretrained("./ner_model")

print("Model and tokenizer saved to ./ner_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./ner_model


### Step 7: Predict & Test

In [53]:
def predict1(text: str):
    # Split the text into words before passing to the tokenizer
    # when is_split_into_words=True
    words = text.split()
    tokens = tokenizer(words,
                       return_tensors="pt",
                       truncation=True,
                       is_split_into_words=True)
    with torch.no_grad():
        outputs = model(**tokens)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
    return {"tokens": tokenizer.tokenize(text), "predictions": predictions}

In [82]:
def predict1(text):

    inputs = tokenizer(text, return_tensors="pt")


    inputs = {k: v.to(model.device) for k, v in inputs.items()}


    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
    predictions = [id2label[p] for p in predictions]
    return predictions

In [84]:
text = "Marwan live in Egypt."
result = predict1(text)
print(result)

['O', 'B-MISC', 'B-PER', 'O', 'O', 'B-LOC', 'O', 'O']


### Step 8: Model Quantization

In [59]:
import os

quantized_model = quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

save_directory = "./quantized_ner_model"
os.makedirs(save_directory, exist_ok=True)

quantized_model.config.to_json_file(os.path.join(save_directory, "config.json"))

torch.save(quantized_model.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

print("Quantized model saved.")

/tmp/ipykernel_1282/3268556610.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quantize_dynamic(


Quantized model saved.


In [70]:
def predict2(text: str):
    words = text.split()
    tokens = tokenizer(words,
                       return_tensors="pt",
                       truncation=True,
                       is_split_into_words=True)

    model.to('cpu')
    tokens = {k: v.to('cpu') for k, v in tokens.items()}


    with torch.no_grad():
        outputs = model(**tokens)

    logits = outputs.logits

    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()


    input_tokens = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])


    predicted_labels = [id2label[p] for p in predictions]

    return list(zip(input_tokens, predicted_labels))



In [85]:
text = "Marwan Live in Egypt."
print(predict2(text))

[('[CLS]', 'O'), ('Mar', 'B-ORG'), ('##wan', 'B-ORG'), ('Live', 'O'), ('in', 'O'), ('Egypt', 'B-LOC'), ('.', 'O'), ('[SEP]', 'O')]


### Step 9: Comparison and Conclusion

In [72]:
import os

def get_model_size(model):
    torch.save(model.state_dict(), "temp.p")
    size = os.path.getsize("temp.p") / (1024 * 1024)
    os.remove("temp.p")
    return size

print(f"Model Size: {get_model_size(quantized_model):.2f} MB")

Model Size: 168.04 MB


In [73]:
import os
import torch

def get_model_size(model, label="Model"):
    torch.save(model.state_dict(), "temp.p")
    size_mb = os.path.getsize("temp.p") / (1024 * 1024)
    os.remove("temp.p")
    print(f"{label} Size: {size_mb:.2f} MB")
    return size_mb

size_before = get_model_size(model, "Standard BERT (Float32)")

size_after = get_model_size(quantized_model, "Quantized BERT (INT8)")

print(f"\nCompression Ratio: {size_before / size_after:.2f}x smaller!")

Standard BERT (Float32) Size: 411.01 MB
Quantized BERT (INT8) Size: 168.04 MB

Compression Ratio: 2.45x smaller!


> Thanls a lot for your Effort